# 00. Data Preparation & Consolidation
## Load all indicators from `nam/` → Create consolidated dataset

**Objective**: 
- Load 8 indicators from `nam/` folder (already updated with Services VA)
- Create consolidated raw dataset: one row = one country-year
- Document transformations for Phase IV questions

**Output**:
- `consolidated_raw_data.csv`: All 8 indicators × 266 countries × time period

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

# Define paths - relative to this notebook location
notebooks_dir = Path('.')
nam_data_dir = Path('../../nam')  # Navigate to nam/ folder
output_dir = Path('./outputs')
output_dir.mkdir(exist_ok=True)

print(f"✓ Paths configured:")
print(f"  NAM Data dir: {nam_data_dir.resolve()}")
print(f"  Output dir: {output_dir.resolve()}")

✓ Paths configured:
  NAM Data dir: E:\WorldBank-Data-Analysis-Project\nam
  Output dir: E:\WorldBank-Data-Analysis-Project\test_while\fix_data\outputs


## 1. Define 8 Indicators (Service Employment → Services VA)

In [2]:
# 8 indicators: Service Employment REPLACED with Services, value added (%)
INDICATORS = {
    "GDP per capita (constant 2015 US$).csv": {
        "label": "GDP per capita",
        "type": "anchor",
        "unit": "USD"
    },
    "Access to electricity (% of population).csv": {
        "label": "Electricity Access",
        "type": "infrastructure",
        "unit": "percent"
    },
    "Individuals using the Internet (% of population).csv": {
        "label": "Internet Penetration",
        "type": "digitalization",
        "unit": "percent"
    },
    "Foreign direct investment, net inflows (% of GDP).csv": {
        "label": "FDI Inflows",
        "type": "integration",
        "unit": "percent"
    },
    "Domestic credit to private sector (% of GDP).csv": {
        "label": "Domestic Credit",
        "type": "finance",
        "unit": "percent"
    },
    "Agriculture, forestry, and fishing, value added (% of GDP).csv": {
        "label": "Agriculture Sector",
        "type": "structure",
        "unit": "percent"
    },
    "Industry (including construction), value added (% of GDP).csv": {
        "label": "Industry Sector",
        "type": "structure",
        "unit": "percent"
    },
    "Services, value added (% of GDP).csv": {
        "label": "Services Sector",
        "type": "structure",
        "unit": "percent"
    }
}

print(f"✓ Defined {len(INDICATORS)} indicators")
print(f"  Note: Service Employment REPLACED with Services, value added (%)")
print()
for filename, info in INDICATORS.items():
    print(f"  • {info['label']:<25} ({info['unit']})")

✓ Defined 8 indicators
  Note: Service Employment REPLACED with Services, value added (%)

  • GDP per capita            (USD)
  • Electricity Access        (percent)
  • Internet Penetration      (percent)
  • FDI Inflows               (percent)
  • Domestic Credit           (percent)
  • Agriculture Sector        (percent)
  • Industry Sector           (percent)
  • Services Sector           (percent)


## 2. Load Data from `nam/` Folder

In [3]:
# Load all indicators
data = {}
load_log = []

for filename, info in INDICATORS.items():
    filepath = nam_data_dir / filename
    
    try:
        df = pd.read_csv(filepath)
        data[info['label']] = df
        
        # Get year columns
        year_cols = [c for c in df.columns if c.startswith('YR')]
        loaded_years = len(year_cols)
        
        load_log.append({
            'indicator': info['label'],
            'file': filename,
            'countries': df.shape[0],
            'years': loaded_years,
            'years_range': f"{int(year_cols[0][2:])}-{int(year_cols[-1][2:])}" if year_cols else "N/A",
            'status': '✓ OK'
        })
    except FileNotFoundError:
        load_log.append({
            'indicator': info['label'],
            'file': filename,
            'status': '✗ NOT FOUND'
        })
        print(f"⚠️ File not found: {filepath}")

load_df = pd.DataFrame(load_log)
print("\n" + "="*80)
print("DATA LOADING SUMMARY")
print("="*80 + "\n")
print(load_df.to_string(index=False))
print(f"\n✓ Loaded {len(data)}/{len(INDICATORS)} indicators")


DATA LOADING SUMMARY

           indicator                                                           file  countries  years years_range status
      GDP per capita                         GDP per capita (constant 2015 US$).csv        266     21   2004-2024   ✓ OK
  Electricity Access                    Access to electricity (% of population).csv        266     21   2004-2024   ✓ OK
Internet Penetration           Individuals using the Internet (% of population).csv        266     21   2004-2024   ✓ OK
         FDI Inflows          Foreign direct investment, net inflows (% of GDP).csv        266     21   2004-2024   ✓ OK
     Domestic Credit               Domestic credit to private sector (% of GDP).csv        266     21   2004-2024   ✓ OK
  Agriculture Sector Agriculture, forestry, and fishing, value added (% of GDP).csv        266     21   2004-2024   ✓ OK
     Industry Sector  Industry (including construction), value added (% of GDP).csv        266     21   2004-2024   ✓ OK
     Serv

## 3. Create Consolidated Dataset

In [4]:
# Create consolidated dataset: one row = one country-year, columns = 8 indicators
first_indicator = list(data.keys())[0]
all_years_cols = [c for c in data[first_indicator].columns if c.startswith('YR')]

# Build master dataframe
consolidated = None

for label, df in data.items():
    # Keep only economy + year columns
    year_cols = [c for c in df.columns if c.startswith('YR')]
    subset = df[['economy'] + year_cols].copy()
    subset = subset.set_index('economy')
    subset.columns = [f"{label}_{c}" for c in subset.columns]  # Rename for clarity
    
    if consolidated is None:
        consolidated = subset.copy()
    else:
        # Merge by country (index)
        consolidated = consolidated.join(subset, how='outer')

print(f"\n✓ Consolidated dataset created:")
print(f"  Shape: {consolidated.shape}")
print(f"  Countries: {len(consolidated)}")
print(f"  Indicators × Years: {consolidated.shape[1]}")
print(f"  Sample columns: {consolidated.columns[:5].tolist()}")
print(f"\n  First 3 rows:")
print(consolidated.head(3))


✓ Consolidated dataset created:
  Shape: (266, 168)
  Countries: 266
  Indicators × Years: 168
  Sample columns: ['GDP per capita_YR2004', 'GDP per capita_YR2005', 'GDP per capita_YR2006', 'GDP per capita_YR2007', 'GDP per capita_YR2008']

  First 3 rows:
         GDP per capita_YR2004  GDP per capita_YR2005  GDP per capita_YR2006  \
economy                                                                        
ABW               29959.774819           29081.706700           28885.911870   
AFE                1271.312757            1314.387938            1364.705254   
AFG                 338.637274             363.640141             367.758312   

         GDP per capita_YR2007  GDP per capita_YR2008  GDP per capita_YR2009  \
economy                                                                        
ABW               29556.838469           29870.664808           26204.059736   
AFE                1417.472432            1440.597284            1414.587751   
AFG                 41

## 4. Save Consolidated Dataset

In [5]:
# Save consolidated data
output_path = output_dir / 'consolidated_raw_data.csv'
consolidated.to_csv(output_path)

print(f"\n✓ Data saved: {output_path}")
print(f"  File size: {output_path.stat().st_size / 1024:.1f} KB")
print(f"\n→ Next: 01_feature_validation.ipynb - Data quality assessment")


✓ Data saved: outputs\consolidated_raw_data.csv
  File size: 574.0 KB

→ Next: 01_feature_validation.ipynb - Data quality assessment
